In [1]:
import os
from tqdm.auto import tqdm
import pandas as pd

# to use data structures
from pydantic import BaseModel, ConfigDict
# from enum import Enum

# OpenAI API
from openai import OpenAI

# Load the API keys from .env
from dotenv import load_dotenv

pd.options.mode.chained_assignment = (
    None  # default='warn' This disables warning of "copying a slice of a DataFrame"
)
tqdm.pandas()  # activate the tqdm for pandas

In [2]:
# load the API keys
load_dotenv(".env")
for api_key in ["OPENAI_API_KEY"]:
    if os.getenv(api_key) is not None:
        print(api_key, "loaded")
    else:
        print(api_key, "missing")
        print("Please create a .env file with the corresponding API key")

OPENAI_API_KEY loaded


In [3]:
from utils.set_paths import (
    path_app_context,
    path_templates,
    path_data,
    path_output,
)

path_batch_files = path_data / "batches"

# import util functions
from utils.content_utils import *
from utils.file_ops import *

In [4]:
summary_excel = pd.read_excel("../EconJM_status.xlsx", sheet_name="sum")
summary_excel

,add_id,good_match,app_status,institution,location,add_rank,add_field,add_category,deadline_target,add_posting,add_how_apply,other_link,need_cover_letter,need_RS,need_TS,need_diversity,OMER_letter_status,KLAUS_letter_status,WOO_letter_status
0,simon_frazer_1,NaN,generating docs,Simon Fraser University,"Burnaby, Canada",assistant,applied micro,research,2025-10-01,EJM,EJM,NaN,1,1,1,1,NaN,NaN,NaN
1,macmaster_1,NaN,generating docs,McMaster University,"Hamilton, Canada",assistant,applied micro,teaching,2025-10-01,EJM,EJM,NaN,1,1,1,1,NaN,NaN,NaN
2,usydney_1,1.0,generating docs,University of Sydney,"Sydney, Asutralia",postdoc,culture,research,2025-11-17,EJM,EJM,NaN,1,0,0,0,NaN,NaN,NaN
3,nova_lisboa_1,NaN,getting info,University Nova Lisboa,"Cascais, Portugal",assistant,environmental,research,2025-11-17,EJM,EJM,NaN,1,0,0,0,NaN,NaN,NaN
4,will_college_1,NaN,getting info,Williams College,"Williamstown, United States",assistant,applied micro,research,2025-11-10,EJM,https://apply.interfolio.com/171825,NaN,1,0,1,0,NaN,NaN,NaN
5,aist_1,1.0,getting info,Institute For Advanced Study,"Touluse, France",fellowship,NaN,research,2025-11-10,NaN,https://www.linkedin.com/posts/nicholascmartin...,NaN,1,0,0,0,NaN,NaN,NaN
6,waseda_1,1.0,getting info,Waseda University,"Waseda, Japan",assistant,applied micro,research,2025-10-10,JOE,https://www.wasedapse.jp/en/fpse2/eng_input.php,https://www.waseda.jp/fpse/pse/news-en/2025/08...,1,1,1,0,NaN,NaN,NaN
7,penn_state_1,1.0,getting info,Pennsylvania State University,"University Park, United States",assistant,NaN,research,2025-11-10,EJM,https://psu.wd1.myworkdayjobs.com/en-US/PSU_Ac...,NaN,1,0,0,0,NaN,NaN,NaN


# Load the needed input files

- Research Statement Template
- Cover Letter Template
- Teaching Statement Template

In [5]:
# Research Statement template
with open(path_templates / "base_text/reseearch_statement.txt", "r") as f:
    lines = f.readlines()

RA_TEMPLATE = " ".join(lines)

# Figure out what docs are needed to generate

In [6]:
columns_to_show = [
    "add_id",
    "institution",
    "location",
    "add_rank",
    "add_field",
    "add_category",
    "need_cover_letter",
    "need_RS",
    "need_TS",
    "need_diversity",
]

In [7]:
completed_docs = [str(d).split("/")[-1] for d in path_output.iterdir() if d.is_dir()]

docs_to_generate = set(summary_excel.add_id.to_list()) - set(completed_docs)

if docs_to_generate:
    print("Need to generate docs for:")
    to_generate_df = summary_excel[summary_excel.add_id.isin(docs_to_generate)][
        columns_to_show
    ]
    docs_list = to_generate_df.astype(str).values.tolist()
    for id in docs_to_generate:
        print(f"- {id}")
else:
    print("No docs needed to generate")

Need to generate docs for:
- macmaster_1
- will_college_1
- waseda_1
- penn_state_1
- nova_lisboa_1
- usydney_1
- aist_1
- simon_frazer_1


In [8]:
to_generate_df.head()

,add_id,institution,location,add_rank,add_field,add_category,need_cover_letter,need_RS,need_TS,need_diversity
0,simon_frazer_1,Simon Fraser University,"Burnaby, Canada",assistant,applied micro,research,1,1,1,1
1,macmaster_1,McMaster University,"Hamilton, Canada",assistant,applied micro,teaching,1,1,1,1
2,usydney_1,University of Sydney,"Sydney, Asutralia",postdoc,culture,research,1,0,0,0
3,nova_lisboa_1,University Nova Lisboa,"Cascais, Portugal",assistant,environmental,research,1,0,0,0
4,will_college_1,Williams College,"Williamstown, United States",assistant,applied micro,research,1,0,1,0


# Add the context and additional data for each prompt

For these all the `raw` documents are transcribed in a few bullet points:

- For Job add use 
    ```
    summarize the job add in a few bullet points. Prioritize the attributes, qualitites they look for in a candidate
    ```
- For the isntitutional values use 
    ```
    summarize the institutions values in a few bullet points
    ```
- For econ research department use 
    ```
    summarize the economics department values and research topics in a few bullet points. Make emphasis on potential co-authors and researcg/teaching groups described there and write it in the file
    ```

In [9]:
# get the add description context
to_generate_df.loc[:, "add_description"] = to_generate_df.apply(
    lambda row: get_txt_info(
        path_app_context / f"{row['add_id']}.txt",
        start_marker="START JOB DESCRIPTION",
        end_marker="END JOB DESCRIPTION",
    ),
    axis=1,
)

to_generate_df.loc[:, "econ_context"] = to_generate_df.apply(
    lambda row: get_txt_info(
        path_app_context / f"{row['add_id']}.txt",
        start_marker="START ECON DEPT",
        end_marker="END ECON DEPT",
    ),
    axis=1,
)

to_generate_df.loc[:, "institution_values"] = to_generate_df.apply(
    lambda row: get_txt_info(
        path_app_context / f"{row['add_id']}.txt",
        start_marker="START INSTITUTIONAL DESCRIPTION",
        end_marker="END INSTITUTIONAL DESCRIPTION",
    ),
    axis=1,
)

# OpenAI Batches

In [10]:
openai_client = OpenAI()
model_name = "gpt-4.1-mini"

In [11]:
# Load the OpenAI batching df
# load history of batches
if (path_batch_files / "batches_history.csv").exists():
    batch_history = pd.read_csv(path_batch_files / "batches_history.csv")
else:
    print("No history of batches found")
    batch_history = pd.DataFrame(
        columns=[
            "created_at",
            "description",
            "model",
            "cat_type",
            "batch_id",
            "status",
            "processing_status",
            "input_file_id",
            "output_file_id",
            "error_file_id",
        ]
    )

batch_history = update_batch_status(batch_history, openai_client)
batch_history.sort_values(["created_at", "model", "cat_type"], inplace=True)
batch_history.to_csv(path_batch_files / "batches_history.csv", index=False)


# Check the inprocess batches
in_process_batches = batch_history[batch_history["status"] == "in_progress"]
print("In-process batches:")
display(in_process_batches.tail(5))

# Check the completed batches
completed_batches = batch_history[
    (batch_history["status"] == "completed")
    & (batch_history["processing_status"] == False)
    & (batch_history["error_file_id"].isna())
]
print("Completed batches:")
display(completed_batches.tail(5))

In-process batches:


/Volumes/jjgecon_dat/Dropbox/Work/Job Search/2025 EconJM/automatization/utils/content_utils.py:247: FutureWarning: Setting an item of incompatible dtype is deprecated and will raise an error in a future version of pandas. Value 'file-EkumtddrxuGoaYEfYPGsym' has dtype incompatible with float64, please explicitly cast to a compatible dtype first.
  batch_df.at[index, "output_file_id"] = batch.output_file_id


,created_at,description,model,cat_type,batch_id,status,processing_status,input_file_id,output_file_id,error_file_id


Completed batches:


,created_at,description,model,cat_type,batch_id,status,processing_status,input_file_id,output_file_id,error_file_id
0,2025-09-10 13:32:59.809253,Generate Research Statements,gpt-4.1-mini,rs,batch_68c1c45ccf6c81908c795d1c4400d1ed,completed,False,file-8GAVWNwMEXZn7iTSrr7NG6,file-EkumtddrxuGoaYEfYPGsym,NaN


# Research Statement

In [12]:
gen_aux = to_generate_df[to_generate_df.need_RS == 1].copy()
gen_aux

,add_id,institution,location,add_rank,add_field,add_category,need_cover_letter,need_RS,need_TS,need_diversity,add_description,econ_context,institution_values
0,simon_frazer_1,Simon Fraser University,"Burnaby, Canada",assistant,applied micro,research,1,1,1,1,- Tenure-track Assistant Professor position in...,"- The department fosters a dynamic, creative, ...",- Student participation and involvement are ce...
1,macmaster_1,McMaster University,"Hamilton, Canada",assistant,applied micro,teaching,1,1,1,1,- Position: Assistant or junior Associate Prof...,- The Department values rigorous training in e...,"- Commitment to creativity, innovation, and ex..."
6,waseda_1,Waseda University,"Waseda, Japan",assistant,applied micro,research,1,1,1,0,,,


In [13]:
def rs_prompt(row):
    prompt = f"""
    You are an expert in writting job application materials for {row.institution}, a {row.add_category} focus institution.
    I want you to modify the following BASE RESEARCH STATEMENT and integrate my fit the institution and department with the INSTITUTION VALUES and ECON DEPARTMENT CONTEXT.
    In addition, highlight my research fit to the JOB ADVERTISEMENT qualities described.
    Keep the length of the research statement to about 800 words and keep the new content as brief as possible.
    Use a professional and academic tone, but make it engaging and easy to read.
    Avoid using hyphens, bullet points, or numbered lists.
    Please output the full research statement as `r_statement` and a short chain-of-thought explanation of how you modified the BASE RESEARCH STATEMENT to fit the institution and department as `cot`.
    The text should follow the Typst formatting in the BASE RESEARCH STATEMENT.

    BASE RESEARCH STATEMENT:
    {RA_TEMPLATE}

    JOB ADVERTISEMENT:
    {row.add_description}

    INSTITUTION VALUES:
    {row.institution_values}

    ECON DEPARTMENT CONTEXT:
    {row.econ_context}
    """
    return prompt

In [14]:
gen_aux.loc[:, "prompt"] = gen_aux.apply(
    lambda row: rs_prompt(row),
    axis=1,
)

In [15]:
class ResearchStatement(BaseModel):
    r_statement: str
    cot: str
    model_config = ConfigDict(extra="forbid")


name_jsonl = "gen_rs"
description_batch = "Generate Research Statements"

content_schema = ResearchStatement
cat_type = "rs"

In [16]:
%run -i batched_chat_gpt.py

Found completed batches for gpt-4.1-mini and RS
Updating the existing old_docs
Need to generate 1 RS docs that contain a using gpt-4.1-mini
Creating a new batch
Created a new batch with id batch_68d97b2cde708190955f1672844df193 and updated the batch history.


# Generate the Docs

Here we assume that all docs have been generated

In [17]:
def generate_docs(df, generated_df, gen_type="rs"):
    if gen_type == "rs":
        dir_name = "research_statement"
        file_name = "Gonzalez_RS.typ"
    else:
        raise ValueError("gen_type not recognized")

    for i, row in df.iterrows():
        print(f"Processing {row.add_id}...")

        new_content = generated_df[
            generated_df.add_id == row.add_id
        ].r_statement_1.values[0]

        # check is the path exists
        target_path = path_output / row.add_id

        if not target_path.exists():
            print(f"Creating directory: {target_path}")
            copy_and_rename_dir(path_templates / f"{dir_name}", target_path, dir_name)
        else:
            print(f"Directory already exists: {target_path}")

        try:
            add_lines_to_typs_doc(target_path / f"{dir_name}/{file_name}", new_content)
            print(f"{file_name} added for {row.add_id}")
        except Exception as e:
            print(f"Error processing {row.add_id}: {e}")

## research statement

In [20]:
rs_gen_df = pd.read_csv(path_output / "rs_docs_gpt-4.1-mini.csv")
rs_gen_df

,add_id,r_statement_1,cot_1
0,macmaster_1,Research Statement\n\nEntertainment media such...,I integrated the BASE RESEARCH STATEMENT with ...
1,simon_frazer_1,"Entertainment media, encompassing television s...",To revise the BASE RESEARCH STATEMENT for Simo...


In [24]:
generate_docs(gen_aux, rs_gen_df, gen_type="rs")

Processing simon_frazer_1...
Directory already exists: ../output_docs/simon_frazer_1
Error processing simon_frazer_1: Placeholder '// INSERT LINES HERE' not found in ../output_docs/simon_frazer_1/research_statement/Gonzalez_RS.typ.
Processing macmaster_1...
Directory already exists: ../output_docs/macmaster_1
Error processing macmaster_1: Placeholder '// INSERT LINES HERE' not found in ../output_docs/macmaster_1/research_statement/Gonzalez_RS.typ.
Processing waseda_1...


IndexError: index 0 is out of bounds for axis 0 with size 0